In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "APTUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,4.729,4.733,4.714,4.716,11987.46,2025-06-01 00:04:59.999999+00:00,56649.58796,381,6860.03,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.000000,0.000000,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,4.717,4.730,4.717,4.728,7763.53,2025-06-01 00:09:59.999999+00:00,36675.27032,267,4323.72,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.000269,0.000150,0.000120,NaN,NaN
2,2025-06-01 00:10:00+00:00,4.728,4.728,4.713,4.715,8694.34,2025-06-01 00:14:59.999999+00:00,41016.26833,290,2987.42,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000060,0.000064,-0.000124,NaN,NaN
3,2025-06-01 00:15:00+00:00,4.716,4.717,4.703,4.711,15320.41,2025-06-01 00:19:59.999999+00:00,72135.00537,414,7443.87,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000353,-0.000077,-0.000275,NaN,NaN
4,2025-06-01 00:20:00+00:00,4.711,4.717,4.704,4.716,9409.25,2025-06-01 00:24:59.999999+00:00,44321.08734,246,4305.75,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000305,-0.000145,-0.000160,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,411
[info] optuna train rows: 53,382
[info] valid rows:        13,346
[info] test rows:         16,683


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-22 18:21:53,775] A new study created in memory with name: no-name-08f6c11d-7b56-4ffb-8f1c-8db65fd0b7ff


[I 2026-03-22 18:21:53,920] Trial 0 finished with value: 0.5316527240103097 and parameters: {'n_estimators': 400, 'learning_rate': 0.09423875899553878, 'max_depth': 5, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 3, 'reg_lambda': 0.13066739238053282, 'scale_pos_weight': 1.390841280949359}. Best is trial 0 with value: 0.5316527240103097.


[I 2026-03-22 18:21:54,059] Trial 1 finished with value: 0.5158682511929822 and parameters: {'n_estimators': 600, 'learning_rate': 0.07036510754376854, 'max_depth': 3, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_lambda': 0.23102018878452935, 'scale_pos_weight': 0.9459356010215937}. Best is trial 0 with value: 0.5316527240103097.


[I 2026-03-22 18:21:54,192] Trial 2 finished with value: 0.5199486673398249 and parameters: {'n_estimators': 400, 'learning_rate': 0.05642937488667569, 'max_depth': 4, 'subsample': 0.7873687420594125, 'colsample_bytree': 0.8447411578889518, 'min_child_weight': 3, 'reg_lambda': 0.3839629299804172, 'scale_pos_weight': 1.0488727806588565}. Best is trial 0 with value: 0.5316527240103097.


[I 2026-03-22 18:21:54,344] Trial 3 finished with value: 0.5195100343855497 and parameters: {'n_estimators': 500, 'learning_rate': 0.0772099153368949, 'max_depth': 3, 'subsample': 0.8542703315240835, 'colsample_bytree': 0.836965827544817, 'min_child_weight': 2, 'reg_lambda': 1.6409286730647923, 'scale_pos_weight': 0.939081516505705}. Best is trial 0 with value: 0.5316527240103097.


...

[I 2026-03-22 18:22:09,428] Trial 96 finished with value: 0.5321223045680264 and parameters: {'n_estimators': 200, 'learning_rate': 0.0553830213537499, 'max_depth': 5, 'subsample': 0.8935493175586058, 'colsample_bytree': 0.8933675051665825, 'min_child_weight': 6, 'reg_lambda': 6.511203293200843, 'scale_pos_weight': 0.9856709149627784}. Best is trial 54 with value: 0.5399445396397988.


[I 2026-03-22 18:22:09,576] Trial 97 finished with value: 0.5350839620319242 and parameters: {'n_estimators': 200, 'learning_rate': 0.046730638896675426, 'max_depth': 5, 'subsample': 0.8611651965507315, 'colsample_bytree': 0.9047288720005815, 'min_child_weight': 3, 'reg_lambda': 7.007856464180281, 'scale_pos_weight': 0.9297706841132949}. Best is trial 54 with value: 0.5399445396397988.


[I 2026-03-22 18:22:09,762] Trial 98 finished with value: 0.5315973001784273 and parameters: {'n_estimators': 200, 'learning_rate': 0.04314642831453679, 'max_depth': 5, 'subsample': 0.8587389368586331, 'colsample_bytree': 0.8833905326700224, 'min_child_weight': 2, 'reg_lambda': 7.320498934033299, 'scale_pos_weight': 1.1682441756752693}. Best is trial 54 with value: 0.5399445396397988.


[I 2026-03-22 18:22:09,921] Trial 99 finished with value: 0.5337750869785496 and parameters: {'n_estimators': 200, 'learning_rate': 0.04650404584403533, 'max_depth': 5, 'subsample': 0.8824586426268837, 'colsample_bytree': 0.9009311384911016, 'min_child_weight': 4, 'reg_lambda': 0.3542161167401387, 'scale_pos_weight': 0.9258408456134986}. Best is trial 54 with value: 0.5399445396397988.


['month_cos', 'hour_cos', 'dow_sin', 'dom_sin', 'range_15', 'hour_sin', 'dist_ma_30', 'mom_30', 'vol_30', 'atr_norm', 'vol_15', 'dom_cos', 'month_sin', 'range_5', 'imbalance_15', 'macd_hist', 'vol_5', 'mom_60', 'trend_strength', 'vol_regime_ratio', 'mom_15', 'range_ratio', 'dow_cos', 'dist_ma_15', 'imbalance_5']
feature
month_cos           11.333786
hour_cos            11.231874
dow_sin             11.122033
dom_sin             11.061129
range_15            11.042221
hour_sin            11.014723
dist_ma_30          10.724579
mom_30              10.710029
vol_30              10.624846
atr_norm            10.594951
vol_15              10.539978
dom_cos             10.483748
month_sin           10.343623
range_5             10.224100
imbalance_15        10.217877
macd_hist           10.052820
vol_5                9.985448
mom_60               9.957398
trend_strength       9.817440
vol_regime_ratio     9.638809
mom_15               9.504341
range_ratio          9.439140
dow_cos           

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

calibrator = artifacts["calibrator"]
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

In [11]:
X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

calibrator = artifacts["calibrator"]

train_pred = calibrator.predict_proba(X_train_full_sel)[:, 1]
test_pred = calibrator.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train ROC AUC:   0.725793
Test ROC AUC:    0.520886
Train PR AUC:    0.696798
Test PR AUC:     0.467765
Train Log Loss:  0.677521
Test Log Loss:   0.687624
Train Brier:     0.242230
Test Brier:      0.247243
Train Accuracy:  0.558671
Test Accuracy:   0.552658
Train Precision: 0.828230
Test Precision:  0.500000
Train Recall:    0.103690
Test Recall:     0.032561
Train F1:        0.184306
Test F1:         0.061140


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret": fwd_ret.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.382, 0.44]  -0.000111   1669  0.006812
(0.44, 0.45]   -0.000219   1668  0.007449
(0.45, 0.456]  -0.000613   1668  0.007714
(0.456, 0.46]  -0.000495   1668  0.007308
(0.46, 0.465]  -0.000369   1669  0.008047
(0.465, 0.469] -0.000411   1668  0.007261
(0.469, 0.474] -0.000408   1668  0.007460
(0.474, 0.479] -0.000083   1668  0.007637
(0.479, 0.487] -0.000079   1668  0.008025
(0.487, 0.536]  0.000881   1669  0.009924


/tmp/ipykernel_979430/1883822384.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret"].mean())
overall_mean_ret = float(eval_df["fwd_ret"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/APTUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/APTUSDT__h6_model.joblib
[saved] features -> models/xgb/APTUSDT__h6_feature_cols.json
[saved] feature importance -> models/xgb/APTUSDT__h6_feature_importance.csv
[saved] metadata -> models/xgb/APTUSDT__h6_meta.json
